# Movie Agent 可复现验收

运行固定输入和消融流程。默认纯仿真；真实成片艺术质量、声音和视觉一致性需要人工盲评。先按运行手册安装依赖和 FFmpeg。

In [ ]:
from pathlib import Path
import subprocess, json, sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
assert (root / "agent" / "service.py").exists(), "请在仓库根目录或 notebooks 目录运行"
python = root / ".venv" / "bin" / "python"
assert python.exists(), "先按 README 创建 .venv"


## 工程回归

包含故障恢复、硬约束、局部修正及音频字幕验证。

In [ ]:
subprocess.run([str(python), "-m", "pytest", "-q"], cwd=root, check=True)


## 12 条输入 × 3 组消融

默认会生成测试视频和参考图。结果单独存储；多人仿真样例可能按预期暴露约束缺口，不伪造修复成功。

In [ ]:
subprocess.run([str(python), "-m", "evaluation.run"], cwd=root, check=True)
report_dir = sorted((root / "evaluation" / "results").iterdir())[-1]
summary = json.loads((report_dir / "summary.json").read_text())
print((report_dir / "summary.md").read_text())


## 检查失败原因

真实服务复测命令见 docs/competition-runbook.md；不会在此 Notebook 默认调用真实服务。

In [ ]:
for row in summary["rows"]:
    if row["actual"] != row["expected"]:
        print(row["case"], row["mode"], row["failure_reason"])
